In [1]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# Load the dataset (downloads metadata and shards if not present locally)
ds = LeRobotDataset("lerobot/aloha_sim_transfer_cube_human")

# Inspect the first frame
sample = ds[0]

print("Dataset successfully loaded!")
print("Available keys in each sample:")
for k in sample.keys():
    v = sample[k]
    shape = getattr(v, "shape", None)
    print(f"  - {k}: shape {shape} | type: {type(v)}")

c:\Users\z00511dv\Downloads\Hackathon\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset successfully loaded!
Available keys in each sample:
  - observation.images.top: shape torch.Size([3, 480, 640]) | type: <class 'torch.Tensor'>
  - observation.state: shape torch.Size([14]) | type: <class 'torch.Tensor'>
  - action: shape torch.Size([14]) | type: <class 'torch.Tensor'>
  - episode_index: shape torch.Size([]) | type: <class 'torch.Tensor'>
  - frame_index: shape torch.Size([]) | type: <class 'torch.Tensor'>
  - timestamp: shape torch.Size([]) | type: <class 'torch.Tensor'>
  - next.done: shape torch.Size([]) | type: <class 'torch.Tensor'>
  - index: shape torch.Size([]) | type: <class 'torch.Tensor'>
  - task_index: shape torch.Size([]) | type: <class 'torch.Tensor'>
  - task: shape None | type: <class 'str'>


c:\Users\z00511dv\Downloads\Hackathon\.venv\Lib\site-packages\torchvision\io\_video_deprecation_warning.py:9: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


In [7]:
import truststore
truststore.inject_into_ssl()

import torch
from torch.utils.data import DataLoader
from lerobot.configs.types import FeatureType, PolicyFeature
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACTPolicy

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

dataset_id = "lerobot/aloha_sim_transfer_cube_human"

# 1. Define ACT parameters
chunk_size = 50   # Predict 50 action steps into the future
fps = 50          # Aloha simulation datasets run at 50 FPS

# Configure the dataset to slice future actions for the chunk
# delta_timestamps tells LeRobotDataset to fetch action[t : t + chunk_size]
delta_timestamps = {
    "action": [i / fps for i in range(chunk_size)],
}

dataset = LeRobotDataset(dataset_id, delta_timestamps=delta_timestamps)

# Verify shape of an action chunk
sample = dataset[0]
print("Loaded action chunk shape:", sample["action"].shape)  # Should be torch.Size([50, 14])

dataloader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
)

# 2. Configure ACT Policy
config = ACTConfig(
    input_features={
        "observation.images.top": PolicyFeature(
            type=FeatureType.VISUAL, shape=(3, 480, 640)
        ),
        "observation.state": PolicyFeature(
            type=FeatureType.STATE, shape=(14,)
        ),
    },
    output_features={
        "action": PolicyFeature(type=FeatureType.ACTION, shape=(14,)),
    },
    chunk_size=chunk_size,
    n_action_steps=chunk_size,
    dim_model=256,
    n_encoder_layers=4,
    n_decoder_layers=4,
)

# 3. Instantiate model
policy = ACTPolicy(config, dataset_stats=dataset.meta.stats)
policy.to(device)
policy.train()

optimizer = torch.optim.AdamW(policy.parameters(), lr=1e-4, weight_decay=1e-4)

# 4. Training loop (quick 60-step test)
total_steps = 2
print(f"Starting ACT training run for {total_steps} steps...")

step = 0
for epoch in range(100):
    for batch in dataloader:
        batch = {
            k: v.to(device) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()
        }

        loss, loss_dict = policy(batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        step += 1
        if step % 10 == 0:
            l1 = loss_dict.get("l1_loss", 0.0)
            kld = loss_dict.get("kld_loss", 0.0)
            print(f"Step {step:2d}/{total_steps} | Loss: {loss.item():.4f} | L1: {l1:.4f} | KLD: {kld:.4f}")

        if step >= total_steps:
            break
    if step >= total_steps:
        break

# 5. Save model
save_path = "checkpoints/act_test_model"
policy.save_pretrained(save_path)
print(f"Model saved to: {save_path}")

Using device: cpu


c:\Users\z00511dv\Downloads\Hackathon\.venv\Lib\site-packages\torchvision\io\_video_deprecation_warning.py:9: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


Loaded action chunk shape: torch.Size([50, 14])


Starting ACT training run for 2 steps...
Model saved to: checkpoints/act_test_model


In [8]:
policy.eval()

# Grab one sample from the dataset as our test observation
sample = dataset[0]

observation = {
    "observation.images.top": sample["observation.images.top"].unsqueeze(0).to(device),
    "observation.state": sample["observation.state"].unsqueeze(0).to(device),
}

with torch.no_grad():
    # select_action handles action chunk prediction and temporal ensembling
    action = policy.select_action(observation)

print("Inference successful!")
print(f"Action shape: {action.shape}")
print(f"Predicted target motor positions (14 joints): \n{action[0].cpu().numpy()}")

Inference successful!
Action shape: torch.Size([1, 14])
Predicted target motor positions (14 joints): 
[ 0.16118246 -0.86636925  1.3263429   0.32067716  0.32057482 -0.5163634
  0.59779286  1.2196679  -0.61739033  1.3067772  -0.9542314  -1.263564
 -1.0505235   0.6767991 ]


In [11]:
import numpy as np
import torch
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw  # if fastdtw is missing: uv pip install fastdtw scipy

policy.eval()

# 1. Grab an entire demonstration episode (e.g., episode 0)
print("Evaluating policy against expert human demonstration...")
episode_indices = [i for i, ep in enumerate(dataset.hf_dataset["episode_index"]) if ep == 0]

ground_truth_actions = []
predicted_actions = []

# Reset policy temporal ensembling queue
policy.reset()

# Sample every 5th frame across the episode for speed
eval_indices = episode_indices[::5]

for idx in eval_indices:
    sample = dataset[idx]
    
    # Ground truth human action at this exact timestep
    # sample["action"] has shape [50, 14], index 0 is the immediate next step
    gt_act = sample["action"][0].numpy()
    ground_truth_actions.append(gt_act)
    
    # Model prediction
    obs = {
        "observation.images.top": sample["observation.images.top"].unsqueeze(0).to(device),
        "observation.state": sample["observation.state"].unsqueeze(0).to(device),
    }
    with torch.no_grad():
        pred_act = policy.select_action(obs).cpu().numpy().squeeze()
    predicted_actions.append(pred_act)

gt_arr = np.array(ground_truth_actions)    # Shape: [T, 14]
pred_arr = np.array(predicted_actions)   # Shape: [T, 14]

# ------------------------------------------------------------------
# 2. Compute Quality Statement Metrics
# ------------------------------------------------------------------

# Metric A: Mean Absolute Error (MAE) & Mean Squared Error (MSE)
mae = np.mean(np.abs(gt_arr - pred_arr))
mse = np.mean((gt_arr - pred_arr) ** 2)

# ------------------------------------------------------------------
# Metric B: Dynamic Time Warping (DTW) Distance (Trajectory Alignment)
# ------------------------------------------------------------------
dtw_distances = []
for joint_idx in range(14):
    # Use scalar absolute difference instead of scipy's euclidean
    dist, _ = fastdtw(gt_arr[:, joint_idx], pred_arr[:, joint_idx], dist=lambda a, b: abs(a - b))
    dtw_distances.append(dist)
avg_dtw = np.mean(dtw_distances)

# Metric C: Jitter / Smoothness Penalty (Second derivative / jerk)
# Jerky actions are unsafe for clinical/cathlab tools
pred_jerk = np.mean(np.abs(np.diff(pred_arr, n=2, axis=0)))

# ------------------------------------------------------------------
# 3. Print the "Trustworthy Quality Statement"
# ------------------------------------------------------------------
print("\n" + "="*50)
print("      AUTOMATED ROBOT QUALITY STATEMENT (v1.0)     ")
print("="*50)
print(f"Evaluated Frames        : {len(eval_indices)} steps")
print(f"Trajectory L1 Error     : {mae:.4f} rad/m (Lower is better)")
print(f"Trajectory MSE          : {mse:.4f} (Lower is better)")
print(f"Avg Dynamic Time Warping: {avg_dtw:.2f} (Path adherence)")
print(f"Action Jitter (Jerk)    : {pred_jerk:.4f} (Safety/smoothness)")
print("-" * 50)

# Safety threshold rule
if pred_jerk < 0.05 and mae < 0.25:
    certification = "PASSED - Safe for Clinical Cathlab Simulation"
else:
    certification = "FLAGGED - High variance/jitter detected; retraining required."
print(f"Deployment Status       : {certification}")
print("="*50)

Evaluating policy against expert human demonstration...



      AUTOMATED ROBOT QUALITY STATEMENT (v1.0)     
Evaluated Frames        : 80 steps
Trajectory L1 Error     : 0.5442 rad/m (Lower is better)
Trajectory MSE          : 0.4218 (Lower is better)
Avg Dynamic Time Warping: 43.53 (Path adherence)
Action Jitter (Jerk)    : 0.0047 (Safety/smoothness)
--------------------------------------------------
Deployment Status       : FLAGGED - High variance/jitter detected; retraining required.


In [6]:
import openpi, sys
print(getattr(openpi, "__file__", None), list(openpi.__path__))
print(sys.executable)

c:\Users\z00511dv\Downloads\Hackathon\.venv\Lib\site-packages\openpi\__init__.py ['c:\\Users\\z00511dv\\Downloads\\Hackathon\\.venv\\Lib\\site-packages\\openpi']
c:\Users\z00511dv\Downloads\Hackathon\.venv\Scripts\python.exe


In [ ]:
import truststore; truststore.inject_into_ssl()   # corporate SSL, as before
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.factory import make_pre_post_processors
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

model_id = "lerobot/smolvla_base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

policy = SmolVLAPolicy.from_pretrained(model_id).to(device).eval()
preprocess, postprocess = make_pre_post_processors(
    policy.config, model_id,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

print(policy.config.input_features)   # which image/state keys it expects

dataset = LeRobotDataset("lerobot/svla_so100_pickplace")
frame = dict(dataset[0])
frame["task"] = "pick up the cube and place it in the box"   # <- the language instruction

policy.reset()
with torch.inference_mode():
    action = postprocess(policy.select_action(preprocess(frame)))

print(action.shape, action)

c:\Users\z00511dv\Downloads\Hackathon\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\z00511dv\Downloads\Hackathon\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\z00511dv\.cache\huggingface\hub\models--lerobot--smolvla_base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In orde

Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...


c:\Users\z00511dv\Downloads\Hackathon\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\z00511dv\.cache\huggingface\hub\models--HuggingFaceTB--SmolVLM2-500M-Video-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!
Xet Storage is enabled for th